NYCPS created the College and Career Readiness metric to replace the Career Readiness Index. The effort tracks the NYSED CCCR College, Career, and (?Community) Readiness value, which will be used for grauation certification. Students are awarded points out of 100 for milestones that align with college and career readiness, such as AP exam scores and apprenticeships.

Load and inspect the school quality dataset. Dataset is not available through the nycschools package, and is large for the SODA Api. Exported manually from 
https://data.cityofnewyork.us/Education/School-Quality-Reports-Data/dnpx-dfnc/

In [2]:
import pandas
from nycschools import schools

In [3]:
school_demographics_df = schools.load_school_demographics()
school_demographics_df.columns

Index(['dbn', 'beds', 'district', 'geo_district', 'boro', 'school_name',
       'short_name', 'ay', 'year', 'school_type', 'school_level',
       'total_enrollment', 'grade_3k', 'grade_pk', 'grade_k', 'grade_1',
       'grade_2', 'grade_3', 'grade_4', 'grade_5', 'grade_6', 'grade_7',
       'grade_8', 'grade_9', 'grade_10', 'grade_11', 'grade_12',
       'non_binary_n', 'non_binary_pct', 'female_n', 'female_pct', 'male_n',
       'male_pct', 'asian_n', 'asian_pct', 'black_n', 'black_pct',
       'hispanic_n', 'hispanic_pct', 'multi_racial_n', 'multi_racial_pct',
       'native_american_n', 'native_american_pct', 'white_n', 'white_pct',
       'missing_race_ethnicity_data_n', 'missing_race_ethnicity_data_pct',
       'swd_n', 'swd_pct', 'ell_n', 'ell_pct', 'poverty_n', 'poverty_pct',
       'eni', 'clean_name', 'zip'],
      dtype='str')

In [4]:
school_quality_df = pandas.read_csv("../data/school_quality.csv")
school_quality_df.columns

Index(['School Year', 'Report Year',
       'District, Borough and School Number (DBN)', 'School Name',
       'Report Type', 'School Type', 'Metric Variable Name',
       'Metric Display Name', 'Number of Students', 'Metric Value',
       'Comparison Group Average', 'Metric Score'],
      dtype='str')

In [5]:
# Data for which unique is sensical shall be described as categorical data.
school_quality_df["School Type"].unique()

<ArrowStringArray>
[          'Elementary',                  'K-8',          'High School',
               'Middle', 'High School Transfer',                  'K-2',
                 'YABC',                  'K-3',                  'K-1',
                  'D75']
Length: 10, dtype: str

In [6]:
school_demographics_df["ay"].unique()

array([2020, 2021, 2022, 2023, 2024, 2017, 2018, 2019, 2016, 2013, 2014,
       2015, 2005, 2006, 2007, 2008, 2009, 2010, 2011])

School quality is a dataset of [1493770 rows x 12 columns] of school quality metrics, by school and year. Metrics are unique by school type. Data collection and validation challenges stemming from the real world experiences of students and educators. The CCR replaces the CRI.

In [7]:
# Get all of the variable display name pairs to lookup in the Educator's Guide. https://infohub.nyced.org/docs/default-source/default-document-library/finalized-2024-25-educator-guide-hs-1.pdf
metrics = school_quality_df[["Metric Variable Name","Metric Display Name"]].drop_duplicates()
ccr_cri_metrics = pandas.concat([metrics[metrics["Metric Variable Name"].str.contains("cri")],metrics[metrics["Metric Variable Name"].str.contains("ccr")]])

In [8]:
ccr_cri_df = school_quality_df[school_quality_df["Metric Variable Name"].isin(ccr_cri_metrics["Metric Variable Name"])]

In [9]:
# merge school quality metrics with demographics
combined_df = pandas.merge(
    left=ccr_cri_df,
    right=school_demographics_df,
    left_on=["District, Borough and School Number (DBN)", "School Year"],
    right_on=["dbn", "ay"]
    )

combined_df.head

<bound method NDFrame.head of        School Year  Report Year District, Borough and School Number (DBN)  \
0             2019         2020                                    17K524   
1             2019         2020                                    17K524   
2             2019         2020                                    17K524   
3             2019         2020                                    17K524   
4             2017         2018                                    02M416   
...            ...          ...                                       ...   
35357         2024         2025                                    32K556   
35358         2024         2025                                    32K564   
35359         2024         2025                                    32K564   
35360         2024         2025                                    32K564   
35361         2024         2025                                    32K564   

                                             

In [21]:
def my_filter(df, mask):
    """Apply mask to df"""
    return df[mask]
schools_with_girls = my_filter(combined_df, combined_df["female_pct"] >= 0.05)
schools_with_girls = schools_with_girls[["ay", "female_pct"]]
schools_with_girls = combined_df.loc[combined_df["female_pct"] >= 0.05, ["ay", "female_pct"]]
schools_with_girls.groupby("ay").median()

,female_pct
ay,
2015,0.489000
2016,0.489000
2017,0.491632
2018,0.492754
2019,0.491857
2023,0.482935
2024,0.478921


In [11]:
schools_with_girls.index

Index([    0,     1,     2,     3,     4,     5,     6,     7,     8,     9,
       ...
       35352, 35353, 35354, 35355, 35356, 35357, 35358, 35359, 35360, 35361],
      dtype='int64', length=35115)

In [14]:
schools_with_girls[schools_with_girls["female_pct"] < 0.33].groupby("ay").median()

,female_pct
ay,
2015,0.223000
2016,0.219000
2017,0.218471
2018,0.228487
2019,0.245353
2023,0.248521
2024,0.221635


In [ ]:
schools_with_girls.sort_values("female_pct")

,ay,school_name,female_pct
25027,2016,A-Tech High School,0.050
25028,2016,A-Tech High School,0.050
25029,2016,A-Tech High School,0.050
26712,2015,Automotive High School,0.056
26710,2015,Automotive High School,0.056
...,...,...,...
30988,2023,"Young Women's Leadership School, Queens",1.000
30983,2023,"Young Women's Leadership School, Queens",1.000
30982,2023,"Young Women's Leadership School, Queens",1.000
23596,2017,"Young Women's Leadership School, Astoria",1.000


In [ ]:
schools_with_girls.describe

<bound method NDFrame.describe of          ay                                        school_name  female_pct
0      2019      International High School at Prospect Heights    0.375676
1      2019      International High School at Prospect Heights    0.375676
2      2019      International High School at Prospect Heights    0.375676
3      2019      International High School at Prospect Heights    0.375676
4      2017                      Eleanor Roosevelt High School    0.604436
...     ...                                                ...         ...
35357  2024  Bushwick Leaders High School for Academic Exce...    0.487465
35358  2024                     Bushwick Community High School    0.441176
35359  2024                     Bushwick Community High School    0.441176
35360  2024                     Bushwick Community High School    0.441176
35361  2024                     Bushwick Community High School    0.441176

[35115 rows x 3 columns]>

Young women do not enroll in tech. This is not a news flash. Can we see an improvement in the data?

In [ ]:
def school_by_type(school_type):
    criterion = school_quality_df["School Type"]==school_type
    return school_quality_df[criterion]

middle_df = school_by_type("Middle")
hs_df = school_by_type("High School")

Filter rows by a column, group by a column, compute a mean

In [ ]:
ccr_df = school_quality_df[school_quality_df["Metric Variable Name"].str.contains("ccr")]
ccr_by_year = ccr_df.groupby(by="Report Year")

For what years have ccr metrics been reported?

In [ ]:
ccr_by_year.count()

,School Year,"District, Borough and School Number (DBN)",School Name,Report Type,School Type,Metric Variable Name,Metric Display Name,Number of Students,Metric Value,Comparison Group Average,Metric Score
Report Year,,,,,,,,,,,
2024,3803,3803,3803,3803,3803,3803,3803,3803,2515,1722,0
2025,4006,4006,4006,4006,4006,4006,4006,4006,2712,1918,922


CCR metrics have been reported for the 2024 and 2025 school year. Metric Score, the normed value relative to Comparison Group, was reported for the first time in 2025.

College and Career Preparatory Course Index
pct_cpci_oa_uc
This metric shows the percentage of students in the school’s four-year cohort who
successfully completed approved rigorous courses and assessments after four
years of high school.

For what years was the pct_cpci_oa_uv reported.

In [ ]:
preparatory_course_index_df = school_quality_df[school_quality_df["Metric Variable Name"]=="pct_cpci_oa_uv"]

In [ ]:
preparatory_course_index_df.head

<bound method NDFrame.head of Empty DataFrame
Columns: [School Year, Report Year, District, Borough and School Number (DBN), School Name, Report Type, School Type, Metric Variable Name, Metric Display Name, Number of Students, Metric Value, Comparison Group Average, Metric Score]
Index: []>

In [ ]:
school_quality_df[school_quality_df["Metric Variable Name"].str.contains("pct_cpci_oa_uc")].groupby(by="Report Year")["Metric Value"].mean()

Report Year
2016    0.070554
2017    0.070786
2018    0.065679
2019    0.070269
2020    0.068727
2021    0.072018
2022    0.080246
2023    0.099860
2024    0.134140
2025    0.138789
Name: Metric Value, dtype: float64

Join on DBN
